# Vérification de l'inférence

Charge BioViL-T et le classifieur, puis compare de vraies paires cohérentes
à des incohérences calibrées.

In [1]:
# Dépendances : encodeur image de BioViL-T (health_multimodal) et lecture des poids (safetensors)
!pip install -q hi-ml-multimodal safetensors


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Chargement des modèles (premier appel : téléchargement de BioViL-T)
from inference import charger_modeles

charger_modeles()
print("Chargement OK")

c:\Users\aicha\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aicha\AppData\Local\Programs\Python\Python313\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 208/208 [00:00<00:00, 3535.42it/s]
[transformers] CXRBertModel LOAD REPORT from: microsoft/BiomedVLP-BioViL-T
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if 

Chargement OK


In [3]:
# Comparaison : vraies paires (cohérentes) contre incohérences calibrées
import pandas as pd, os, statistics as st
from inference import predire

def trouver(nom):
    for base in ["../2_preparation_donnees", ".", "../1_exploration", ".."]:
        p = os.path.join(base, nom)
        if os.path.exists(p):
            return p
    return None

MATCHES = trouver("chexpert_matches_sample_2.csv")
MISMATCHES = trouver("chexpert_mismatches_swap_cluster.csv")
IMG_BASE = "../CheXpert/CheXpert-v1.0-small"
print("matches   :", MATCHES)
print("mismatches:", MISMATCHES, "\n")

def echantillon(csv, n=5):
    df = pd.read_csv(csv, usecols=["path_to_image", "report"])
    out = []
    for rel, rep in zip(df["path_to_image"].astype(str), df["report"].astype(str)):
        if "frontal" not in rel.lower() or len(rep.strip()) < 20:
            continue
        p = os.path.join(IMG_BASE, rel)
        if os.path.exists(p):
            out.append((p, rep))
        if len(out) >= n:
            break
    return out

print("Calcul sur CPU, patiente quelques minutes...\n")
pc = [predire(i, r)["probabilite"] for i, r in echantillon(MATCHES)]
pi = [predire(i, r)["probabilite"] for i, r in echantillon(MISMATCHES)]

for x in pc:
    print(f"cohérente    prob={x:.3f}  -> {'COHÉRENT' if x >= 0.5 else 'INCOHÉRENT'}")
print()
for x in pi:
    print(f"incohérente  prob={x:.3f}  -> {'COHÉRENT' if x >= 0.5 else 'INCOHÉRENT'}")

print(f"\nMoyenne cohérentes   : {st.mean(pc):.3f}  (on l'attend haute)")
print(f"Moyenne incohérentes : {st.mean(pi):.3f}  (on l'attend basse)")

matches   : ../2_preparation_donnees\chexpert_matches_sample_2.csv
mismatches: ../2_preparation_donnees\chexpert_mismatches_swap_cluster.csv 

Calcul sur CPU, patiente quelques minutes...

cohérente    prob=0.816  -> COHÉRENT
cohérente    prob=0.870  -> COHÉRENT
cohérente    prob=0.889  -> COHÉRENT
cohérente    prob=0.981  -> COHÉRENT
cohérente    prob=0.909  -> COHÉRENT

incohérente  prob=0.013  -> INCOHÉRENT
incohérente  prob=0.037  -> INCOHÉRENT
incohérente  prob=0.014  -> INCOHÉRENT
incohérente  prob=0.022  -> INCOHÉRENT
incohérente  prob=0.490  -> INCOHÉRENT

Moyenne cohérentes   : 0.893  (on l'attend haute)
Moyenne incohérentes : 0.115  (on l'attend basse)
